# 06 · Summaries, an LLM test and a complaints assistant

Three uses of a large language model (LLM), each checked rather than trusted:

1. **Name and summarise each topic** from notebook 05, so a manager reads "Frozen accounts after Zelle
   payments" instead of a list of words.
2. **LLM vs the classic model:** can an LLM route complaints without any training? We compare it with the
   TF-IDF model from notebook 04 on the same complaints, including speed.
3. **A complaints assistant:** ask a question in plain English; it finds the most relevant complaints and
   answers using only those, citing complaint IDs. We check every citation is real.

**Which LLM:** on Databricks, a Foundation Model endpoint called through `ai_query()`; locally and in GitHub
Actions, the open model Qwen2.5-3B-Instruct on the CPU. No API keys anywhere.

In [1]:
from radar import setup, run_sql, save_table, llm, llm_name, mlflow_setup
setup()
print("LLM:", llm_name())

LLM: Qwen/Qwen2.5-3B-Instruct


## Part 1 · Name and summarise the topics
### Load the topics and their most typical complaints

In [2]:
topics = run_sql("SELECT * FROM topics ORDER BY topic")
data = run_sql("SELECT * FROM topic_examples ORDER BY topic, rank")
print(f"{len(topics)} topics, {len(data)} example complaints")

20 topics, 100 example complaints


### The prompt
We give the model the topic's top words and its 4 most typical complaints (cut to 450 characters each), and ask
for a short name and a two-sentence summary, as JSON so the answer is easy to read back.

In [3]:
def topic_prompt(words: str, complaints: list) -> str:
    shown = "\n\n".join(f"Complaint {i + 1}: {c[:450]}" for i, c in enumerate(complaints[:4]))
    return (
        "You are an analyst at a bank reviewing customer complaints sent to the US CFPB.\n"
        f"These complaints share one theme. Key words: {words}.\n\n{shown}\n\n"
        "Reply with JSON only, no other text, in exactly this form:\n"
        '{"name": "<3 to 6 word name for the theme>", "summary": "<2 sentences: what customers are complaining about and what went wrong>"}'
    )

### Read the model's JSON safely
Small models sometimes add text around the JSON. We take the part between the first `{` and the last `}`.

In [4]:
import json, re

def read_json(text: str) -> dict:
    match = re.search(r"\{.*\}", text, re.S)
    try:
        return json.loads(match.group(0)) if match else {}
    except json.JSONDecodeError:
        return {}

### Ask for every topic

In [5]:
import time
rows = []
for _, t in topics.iterrows():
    examples = data[data["topic"] == t["topic"]]["narrative"].tolist()
    start = time.time()
    answer = read_json(llm(topic_prompt(t["top_words"], examples), max_tokens=160))
    rows.append({"topic": t["topic"], "name": answer.get("name") or f"Topic {t['topic']}",
                 "summary": answer.get("summary") or "", "seconds": round(time.time() - start, 1)})
    print(t["topic"], "·", rows[-1]["name"], f"({rows[-1]['seconds']}s)", flush=True)
import pandas as pd
data = pd.DataFrame(rows)
data

/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files:  50%|█████     | 1/2 [00:17<00:17, 17.63s/it]

Fetching 2 files: 100%|██████████| 2/2 [00:23<00:00, 10.76s/it]

Fetching 2 files: 100%|██████████| 2/2 [00:23<00:00, 11.79s/it]

... (50 outputs left out)

17 · Deposit Hold Issues (46.2s)


18 · Notification and Signature Issues (47.1s)


19 · Privacy Violations (52.8s)


,topic,name,summary,seconds
0,0,Financial Documentation Issues,Customers complain about inadequate financial ...,158.8
1,1,Platform Fraud & Inadequate Protection,Customers complain about Cash App's failure to...,54.2
2,2,Credit Reporting Disputes,Customers are disputing inaccurate and unverif...,55.1
3,3,Zelle Dispute Resolution Issues,Customers report difficulties in recovering lo...,52.2
4,4,Identity Theft Complaints,Customers complain about fraudulent accounts a...,43.0
5,5,Debt Validation Requests,Customers are complaining about not receiving ...,59.3
6,6,Debt Collector Misreporting,Customers are complaining that debt collectors...,65.0
7,7,Escrow Issues and Servicing Problems,Customers complain about incorrect escrow anal...,53.7
8,8,Wells Fargo Account Fraud,Customers complain about unauthorized account ...,40.3
9,9,Overdraft Fees and Lagged Posting,Customers are complaining about being charged ...,59.3


In [6]:
save_table(data.drop(columns="seconds"), "topic_names")

'topic_names: 20 rows'

## Part 2 · Can an LLM route complaints without training?
### Take the same test complaints the classic model saw
We use 120 complaints from notebook 04's test set (the LLM is slow on a CPU, so not all of them).

In [7]:
data = run_sql("""
    SELECT p.complaint_id, p.product, p.predicted AS tfidf_predicted, c.narrative
    FROM predictions p JOIN complaints c ON c.complaint_id = p.complaint_id
""")
data = data.sample(min(120, len(data)), random_state=11).reset_index(drop=True)
PRODUCTS = sorted(run_sql("SELECT DISTINCT product FROM complaints")["product"])
PRODUCTS

['Bank account',
 'Credit card',
 'Credit reporting',
 'Debt collection',
 'Debt management',
 'Money transfer',
 'Mortgage',
 'Personal loan',
 'Prepaid card',
 'Student loan',
 'Vehicle loan']

### The prompt
The model sees the list of products and one complaint, and must answer with one product name.

In [8]:
def route_prompt(narrative: str) -> str:
    return ("Which product is this US consumer complaint about? Choose exactly one from this list:\n"
            + "\n".join(f"- {p}" for p in PRODUCTS)
            + f"\n\nComplaint: {narrative[:700]}\n\nAnswer with the product name only.")

def to_product(answer: str) -> str:
    """Match the model's answer to a product name; 'unknown' if it named none."""
    answer = answer.lower()
    hits = [p for p in PRODUCTS if p.lower() in answer]
    return max(hits, key=len) if hits else "unknown"

### Run it and time it

In [9]:
answers, seconds = [], []
for text in data["narrative"]:
    start = time.time()
    answers.append(to_product(llm(route_prompt(text), max_tokens=12)))
    seconds.append(time.time() - start)
    if len(seconds) % 20 == 0:
        print(f"{len(seconds)} done, {sum(seconds) / len(seconds):.1f}s each", flush=True)
data["llm_predicted"] = answers
data["llm_seconds"] = seconds
data[["product", "tfidf_predicted", "llm_predicted"]].head(10)

20 done, 11.3s each


40 done, 11.1s each


60 done, 11.2s each


80 done, 11.3s each


100 done, 11.3s each


120 done, 11.3s each


,product,tfidf_predicted,llm_predicted
0,Debt collection,Debt collection,Credit card
1,Credit reporting,Credit reporting,Credit reporting
2,Bank account,Bank account,Bank account
3,Mortgage,Mortgage,Mortgage
4,Debt collection,Debt collection,Student loan
5,Debt collection,Debt collection,Credit card
6,Personal loan,Vehicle loan,Vehicle loan
7,Student loan,Student loan,Student loan
8,Credit card,Bank account,unknown
9,Debt management,Bank account,Bank account


### Compare
Same 120 complaints, same right answers. The classic model was trained on our data; the LLM saw only the list.

In [10]:
from sklearn.metrics import accuracy_score, f1_score
comparison = pd.DataFrame([
    {"model": "TF-IDF + logistic regression (trained)", "accuracy": accuracy_score(data["product"], data["tfidf_predicted"]),
     "macro_f1": f1_score(data["product"], data["tfidf_predicted"], average="macro", labels=PRODUCTS), "seconds_per_complaint": None},
    {"model": f"{llm_name()} (zero-shot)", "accuracy": accuracy_score(data["product"], data["llm_predicted"]),
     "macro_f1": f1_score(data["product"], data["llm_predicted"], average="macro", labels=PRODUCTS),
     "seconds_per_complaint": data["llm_seconds"].mean()},
]).round(4)
comparison

,model,accuracy,macro_f1,seconds_per_complaint
0,TF-IDF + logistic regression (trained),0.8167,0.6988,NaN
1,Qwen/Qwen2.5-3B-Instruct (zero-shot),0.4417,0.3764,11.2951


### Log the comparison in MLflow

In [11]:
mlflow = mlflow_setup()
with mlflow.start_run(run_name="llm_zero_shot_routing"):
    mlflow.log_params({"llm": llm_name(), "complaints": len(data)})
    mlflow.log_metrics({"llm_accuracy": comparison.iloc[1]["accuracy"], "llm_macro_f1": comparison.iloc[1]["macro_f1"],
                        "tfidf_accuracy": comparison.iloc[0]["accuracy"], "tfidf_macro_f1": comparison.iloc[0]["macro_f1"],
                        "llm_seconds_per_complaint": data["llm_seconds"].mean()})
save_table(comparison, "llm_comparison")
save_table(data.drop(columns=["narrative"]), "llm_predictions")

'llm_predictions: 120 rows'

## Part 3 · A complaints assistant
### Step 1: find the relevant complaints (retrieval)
Before the LLM answers, we find the 6 complaints most similar to the question, using TF-IDF and cosine
similarity (how much two texts share the same important words). The LLM then answers **only** from those 6.
This is retrieval-augmented generation (RAG) in its simplest form.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

data = run_sql("SELECT complaint_id, product, date_received, text, narrative FROM complaints")
index_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.5, sublinear_tf=True)
index = index_vec.fit_transform(data["text"])

def retrieve(question: str, k: int = 6) -> pd.DataFrame:
    scores = linear_kernel(index_vec.transform([question.lower()]), index).ravel()
    best = scores.argsort()[::-1][:k]
    return data.iloc[best].assign(score=scores[best])

retrieve("Zelle scam bank refused refund")[["complaint_id", "product", "score"]]

,complaint_id,product,score
20678,18863856,Prepaid card,0.292969
25255,22995087,Personal loan,0.257060
25523,23252453,Bank account,0.232103
23712,21499879,Money transfer,0.222254
11651,14448169,Prepaid card,0.160924
14626,15545907,Bank account,0.158016


### Step 2: answer from those complaints, with citations

In [13]:
def answer_prompt(question: str, found: pd.DataFrame) -> str:
    shown = "\n\n".join(f"[{r.complaint_id}] ({r.product}) {r.narrative[:600]}" for r in found.itertuples())
    return ("You answer questions about consumer complaints using ONLY the complaints below. "
            "Cite the complaint IDs you use in square brackets, like [1234567]. "
            "If the complaints don't answer the question, say so. Keep it under 120 words.\n\n"
            f"{shown}\n\nQuestion: {question}")

def ask(question: str) -> dict:
    found = retrieve(question)
    text = llm(answer_prompt(question, found), max_tokens=220)
    cited = set(re.findall(r"\[(\d+)\]", text))
    real = set(found["complaint_id"].astype(str))
    return {"question": question, "answer": text, "sources": found["complaint_id"].astype(str).tolist(),
            "cited": sorted(cited), "made_up_citations": sorted(cited - real)}

### Ask a few questions a complaints manager would ask
We check each answer: every complaint ID it cites must be one of the 6 it was given. A made-up ID means the
model invented something.

In [14]:
QUESTIONS = [
    "What happens to customers who are scammed through Zelle and ask their bank for a refund?",
    "Why do customers say their bank account was frozen or closed?",
    "What problems do people describe with mortgage escrow and property taxes?",
    "How do debt collectors contact people about debts they say they don't owe?",
    "What goes wrong when people dispute errors on their credit report?",
    "What do customers say about credit card late fees and interest charges?",
]
rows = []
for q in QUESTIONS:
    start = time.time()
    rows.append(ask(q))
    print(f"{time.time() - start:.0f}s · {q}", flush=True)
data = pd.DataFrame(rows)
data[["question", "cited", "made_up_citations"]]

73s · What happens to customers who are scammed through Zelle and ask their bank for a refund?


105s · Why do customers say their bank account was frozen or closed?


139s · What problems do people describe with mortgage escrow and property taxes?


77s · How do debt collectors contact people about debts they say they don't owe?


87s · What goes wrong when people dispute errors on their credit report?


88s · What do customers say about credit card late fees and interest charges?


,question,cited,made_up_citations
0,What happens to customers who are scammed thro...,[11554277],[]
1,Why do customers say their bank account was fr...,"[12000290, 14753248, 15181999, 15208217, 15218...",[]
2,What problems do people describe with mortgage...,"[11786617, 14902248, 18071972, 18555238, 20609...",[]
3,How do debt collectors contact people about de...,"[11739514, 12277468, 22760697]",[]
4,What goes wrong when people dispute errors on ...,"[11339615, 14373360]",[]
5,What do customers say about credit card late f...,"[11374603, 16953308]",[]


In [15]:
print(f"answers with a made-up citation: {(data['made_up_citations'].str.len() > 0).sum()} of {len(data)}")
print(f"answers citing at least one complaint: {(data['cited'].str.len() > 0).sum()} of {len(data)}")
data["sources"] = data["sources"].map(",".join)
data["cited"] = data["cited"].map(",".join)
data["made_up_citations"] = data["made_up_citations"].map(",".join)
save_table(data, "assistant_answers")

answers with a made-up citation: 0 of 6
answers citing at least one complaint: 6 of 6


'assistant_answers: 6 rows'